# Phase 3 — hardware tier comparison

Single-request long-context scaling for **Llama-3.1-8B-Instruct** across hardware tiers (T4 16GB, L4 24GB, A100 40GB, H100 80GB).

**How to run:** switch the Colab runtime to the desired GPU, then run the matching cell. Each cell records to a hardware-specific JSONL under `results/`. The final cells load every `results/phase3_*.jsonl` and produce a comparison plot.

Scope is intentionally narrow: one model, batch size 1, four context lengths. The point is to show how the OOM frontier and latency curves move with GPU memory and bandwidth, not to redo Phases 1, 2, or 4 on every tier.

In [ ]:
import os

REPO_URL = "https://github.com/sonavk2/LLM_Inference.git"

if not os.path.exists("results"):
    !git clone {REPO_URL}
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    %cd {repo_name}

In [ ]:
from huggingface_hub import login
login()

## A100 sweep

Runtime → A100 GPU. Reuses Phase 1's A100 result file if it already exists; otherwise re-runs the sweep here.

In [ ]:
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --hardware nvidia_a100 \
  --results-path results/phase3_llama31_a100.jsonl

## L4 sweep

Runtime → L4 GPU (24 GB). Llama-3.1-8B in bf16 fits, but KV cache headroom is tight — expect OOM at 32k or 64k.

In [ ]:
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --hardware nvidia_l4 \
  --results-path results/phase3_llama31_l4.jsonl

## H100 sweep

Runtime → H100 GPU (80 GB). Plenty of headroom; this is the upper-bound reference point.

In [ ]:
!python3 scripts/run_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --hardware nvidia_h100 \
  --results-path results/phase3_llama31_h100.jsonl

## T4 — recorded as load-OOM rows

T4 has 16 GB and Llama-3.1-8B bf16 weights are ~16 GB, so the model can't load before any KV cache is allocated. This cell records that finding directly without running the sweep (matches the approach used in `phase1_baseline.ipynb`). Switch to a T4 runtime first if you want to verify the OOM live; otherwise just run this cell on any GPU.

In [ ]:
import json

err = ('CUDA OOM at model load: bf16 weights ~16 GB exceed T4 free memory '
       '(14.56 GB total). Llama-3.1-8B in bf16 cannot load on T4; INT8 or '
       'INT4 quantization is required before any context-length sweep.')

with open('results/phase3_llama31_t4.jsonl', 'w') as f:
    for ctx in [8192, 16384, 32768, 65536]:
        f.write(json.dumps({
            'model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct',
            'backend': 'huggingface',
            'hardware': 'nvidia_t4',
            'context_length': ctx, 'batch_size': 1, 'max_new_tokens': 128,
            'ttft_seconds': None, 'tpot_seconds': None,
            'total_latency_seconds': None, 'tokens_per_second': None,
            'peak_gpu_memory_gb': None, 'kv_cache_memory_gb': None,
            'success': False, 'error': err,
            'prompt_format': 'synthetic_repeat',
            'is_native_context': ctx <= 32768,
            'image_token_count': None, 'text_token_count': None,
            'quantization': None,
        }) + '\n')
print('T4 finding recorded as 4 load-OOM rows in results/phase3_llama31_t4.jsonl')

## Cross-tier comparison

Loads every `results/phase3_*.jsonl` and produces:
- A summary table (one row per hardware × context).
- A 2×3 metrics plot: TTFT, TPOT, throughput, total latency, peak GPU memory, KV-cache estimate vs. context length, one line per hardware tier.
- A memory-frontier matrix (hardware × context, green = OK, red = OOM).

In [ ]:
import json, glob
from pathlib import Path
import pandas as pd

rows = []
for path in sorted(glob.glob('results/phase3_*.jsonl')):
    for line in open(path):
        r = json.loads(line)
        r['_source'] = Path(path).name
        rows.append(r)
if not rows:
    raise SystemExit('No results/phase3_*.jsonl files found yet.')

df = pd.DataFrame(rows)

TIER_ORDER = ['nvidia_t4', 'nvidia_l4', 'nvidia_a100', 'nvidia_h100']
def tier_key(hw):
    return TIER_ORDER.index(hw) if hw in TIER_ORDER else len(TIER_ORDER)
df['tier_rank'] = df['hardware'].map(tier_key)

print(f'loaded {len(df)} rows from {df["_source"].nunique()} files')
print(df[['hardware', 'context_length', 'success', 'ttft_seconds',
          'total_latency_seconds', 'peak_gpu_memory_gb', 'kv_cache_memory_gb']]
      .sort_values(['tier_rank', 'context_length'])
      .to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

PLOT_DIR = Path('results/plots'); PLOT_DIR.mkdir(parents=True, exist_ok=True)
METRICS = [
    ('ttft_seconds',          'TTFT (s)'),
    ('tpot_seconds',          'TPOT (s)'),
    ('tokens_per_second',     'Throughput (tok/s)'),
    ('total_latency_seconds', 'Total latency (s)'),
    ('peak_gpu_memory_gb',    'Peak GPU memory (GB)'),
    ('kv_cache_memory_gb',    'KV-cache estimate (GB)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Phase 3 — Llama-3.1-8B single-request scaling across GPU tiers',
             fontsize=14, fontweight='bold')
for (col, ylabel), ax in zip(METRICS, axes.flat):
    for hw, grp in sorted(df.groupby('hardware'), key=lambda kv: tier_key(kv[0])):
        ok   = grp[grp.success == True ].sort_values('context_length')
        fail = grp[grp.success == False].sort_values('context_length')
        if len(ok):
            ax.plot(ok.context_length, ok[col], marker='o', label=hw)
        if len(fail):
            y = [0] * len(fail)
            ax.scatter(fail.context_length, y, marker='x', color='red', s=120, zorder=5, linewidths=2.5)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('context length (tokens)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(labels), bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=(0, 0.04, 1, 0.97))
out = PLOT_DIR / 'phase3_hardware_comparison.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

In [ ]:
import numpy as np
from matplotlib.patches import Rectangle

pivot_ok = df.pivot_table(index='hardware', columns='context_length',
                          values='success', aggfunc='first')
pivot_ok = pivot_ok.reindex([t for t in TIER_ORDER if t in pivot_ok.index])

fig, ax = plt.subplots(figsize=(1.6 * len(pivot_ok.columns) + 2,
                                 0.9 * len(pivot_ok.index) + 1))
for i, hw in enumerate(pivot_ok.index):
    for j, ctx in enumerate(pivot_ok.columns):
        v = pivot_ok.loc[hw, ctx]
        color = '#bde0a8' if v == True else ('#f4a8a8' if v == False else '#dddddd')
        ax.add_patch(Rectangle((j, i), 1, 1, facecolor=color, edgecolor='white'))
        label = 'OK' if v == True else ('OOM' if v == False else '-')
        ax.text(j + 0.5, i + 0.5, label, ha='center', va='center', fontsize=10)
ax.set_xlim(0, len(pivot_ok.columns)); ax.set_ylim(0, len(pivot_ok.index))
ax.invert_yaxis()
ax.set_xticks(np.arange(len(pivot_ok.columns)) + 0.5); ax.set_xticklabels(pivot_ok.columns)
ax.set_yticks(np.arange(len(pivot_ok.index)) + 0.5);   ax.set_yticklabels(pivot_ok.index)
ax.set_xlabel('context length (tokens)')
ax.set_title('Phase 3 — memory frontier across GPU tiers (Llama-3.1-8B)')
ax.tick_params(length=0)
for spine in ax.spines.values(): spine.set_visible(False)
plt.tight_layout()
out = PLOT_DIR / 'phase3_frontier.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()